In [1]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 4.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 37.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 33.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.8 MB/s eta 0:00:0000:01m00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 2.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 20.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 10.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 36.8 MB/s eta 0:00:0000:0100:01
  Attempting uninstall

In [6]:
import os
import json
from torch.utils.data import Dataset
import tqdm
import matplotlib.pyplot as plt
import numpy as np
import random
import torch
import matplotlib.patches as patches
from torchvision import transforms
from scipy.io import loadmat
import cv2
from torch.utils.data import Dataset, DataLoader
from ultralytics import YOLO

class AverageMeter(object):
    """ Computes and stores the average and current value.
    Can handle both scalar values and numpy arrays.
    """
    def __init__(self, unit='-', is_vector=False):
        self.reset()
        self.unit = unit
        self.is_vector = is_vector

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0
        
    def update(self, val, n=1):
        self.val = val
        
        # Handle initialization for vectors
        if self.count == 0 and isinstance(val, np.ndarray):
            self.sum = np.zeros_like(val, dtype=np.float64)
            
        # Update sum and compute average
        if isinstance(val, np.ndarray):
            self.sum = self.sum + val * n
        else:
            self.sum += val * n
            
        self.count += n
        
        if self.count > 0:
            if isinstance(self.sum, np.ndarray):
                self.avg = self.sum / self.count
            else:
                self.avg = self.sum / self.count
                
    def magnitude(self):
        """Return magnitude for vector quantities"""
        if isinstance(self.avg, np.ndarray):
            return np.linalg.norm(self.avg)
        return self.avg

def calculate_iou(box1, box2):
    """
    Calculate the Intersection over Union (IoU) between two bounding boxes.
    
    Args:
        box1: [xmin, xmax, ymin, ymax] format (ground truth)
        box2: [xmin, ymin, xmax, ymax] format (detection format)
    
    Returns:
        IoU value
    """
    # Convert box2 from [xmin, ymin, xmax, ymax] to [xmin, xmax, ymin, ymax]
    box2_converted = [box2[0], box2[2], box2[1], box2[3]]
    
    # Calculate intersection area
    x_min_inter = max(box1[0], box2_converted[0])
    y_min_inter = max(box1[2], box2_converted[2])
    x_max_inter = min(box1[1], box2_converted[1])
    y_max_inter = min(box1[3], box2_converted[3])
    
    if x_max_inter < x_min_inter or y_max_inter < y_min_inter:
        return 0.0  # No intersection
    
    intersection = (x_max_inter - x_min_inter) * (y_max_inter - y_min_inter)
    
    # Calculate union area
    box1_area = (box1[1] - box1[0]) * (box1[3] - box1[2])
    box2_area = (box2_converted[1] - box2_converted[0]) * (box2_converted[3] - box2_converted[2])
    union = box1_area + box2_area - intersection
    
    return intersection / union if union > 0 else 0.0

class SPEEDDataset(Dataset):
    def __init__(self, images_dir, json_dir, tango_points_dir, camera_matrix_dir, transform=None, is_train=True):
        self.image_width = 1920
        self.image_height = 1200
        self.aspect_ratio = self.image_width * 1.0 / self.image_height
        self.pixel_std = 200
        self.num_joints = 11
        self.image_size = [640, 640]
        self.scale_factor = 0.25
        self.rotation_factor = 30
        self.sigma = 2
        self.is_train = is_train
        self.transform = transform

        self.imagesList = []
        self.frameInfoList = []
        self.images_dir = images_dir
        self.json_dir = json_dir

        self.keypts3d = self.load_tango_3d_keypoints(tango_points_dir) # https://www.desmos.com/3d/jud6lng9gn
        self.cameraMatrix, self.distCoeffs = self.load_camera_intrinsics(camera_matrix_dir)

        with open(self.json_dir, 'r') as f:
            annotations = json.load(f)
            lookup = { item['filename']: item for item in annotations }
            cnt = 0
            for filename in tqdm.tqdm(sorted(os.listdir(self.images_dir))):
                if filename not in lookup:
                    continue
                frame_idx = int(''.join(filter(str.isdigit, filename.split('.')[0])))
                self.imagesList.append(os.path.join(self.images_dir, filename))

                q_vbs2tango = np.array(lookup[filename]["q_vbs2tango"], dtype=np.float32)
                r_Vo2To_vbs = np.array(lookup[filename]['r_Vo2To_vbs_true'], dtype=np.float32)

                frame_info = {
                    'filename': filename,
                    'q_vbs2tango': q_vbs2tango,
                    'r_Vo2To_vbs': r_Vo2To_vbs
                }
                self.frameInfoList.append(frame_info)

                cnt = cnt + 1
                # if cnt>0:
                #     break

    def __len__(self):
        return len(self.imagesList)
    
    def quat2dcm(self, q):
        """ Computing direction cosine matrix from quaternion, adapted from PyNav.
        Arguments:
            q: (4,) numpy.ndarray - unit quaternion (scalar-first)
        Returns:
            dcm: (3,3) numpy.ndarray - corresponding DCM
        """

        # normalizing quaternion
        q = q / np.linalg.norm(q)

        q0, q1, q2, q3 = q[0], q[1], q[2], q[3]
        dcm = np.zeros((3, 3))
        dcm[0, 0] = 2 * q0 ** 2 - 1 + 2 * q1 ** 2
        dcm[1, 1] = 2 * q0 ** 2 - 1 + 2 * q2 ** 2
        dcm[2, 2] = 2 * q0 ** 2 - 1 + 2 * q3 ** 2
        dcm[0, 1] = 2 * q1 * q2 + 2 * q0 * q3
        dcm[0, 2] = 2 * q1 * q3 - 2 * q0 * q2
        dcm[1, 0] = 2 * q1 * q2 - 2 * q0 * q3
        dcm[1, 2] = 2 * q2 * q3 + 2 * q0 * q1
        dcm[2, 0] = 2 * q1 * q3 + 2 * q0 * q2
        dcm[2, 1] = 2 * q2 * q3 - 2 * q0 * q1

        return dcm

    def load_tango_3d_keypoints(self, mat_dir):
        vertices = loadmat(mat_dir)['tango3Dpoints']
        corners3D = np.transpose(np.array(vertices, dtype=np.float32))
        return corners3D

    def load_camera_intrinsics(self, camera_json):
        with open(camera_json) as f:
            cam = json.load(f)
        cameraMatrix = np.array(cam['cameraMatrix'], dtype=np.float32)
        distCoeffs = np.array(cam['distCoeffs'], dtype=np.float32)
        return cameraMatrix, distCoeffs

    def project_keypoints(self, q_vbs2tango, r_Vo2To_vbs, cameraMatrix, distCoeffs, keypoints):
        if keypoints.shape[0] != 3:
            keypoints = np.transpose(keypoints)
        keypoints = np.vstack((keypoints, np.ones((1, keypoints.shape[1]))))
        pose_mat = np.hstack((np.transpose(self.quat2dcm(q_vbs2tango)),
                              np.expand_dims(r_Vo2To_vbs, 1)))
        xyz = np.dot(pose_mat, keypoints)
        x0, y0 = xyz[0, :] / xyz[2, :], xyz[1, :] / xyz[2, :]
        r2 = x0 * x0 + y0 * y0
        cdist = 1 + distCoeffs[0] * r2 + distCoeffs[1] * r2 * r2 + distCoeffs[4] * r2 * r2 * r2
        x = x0 * cdist + distCoeffs[2] * 2 * x0 * y0 + distCoeffs[3] * (r2 + 2 * x0 * x0)
        y = y0 * cdist + distCoeffs[2] * (r2 + 2 * y0 * y0) + distCoeffs[3] * 2 * x0 * y0
        points2D = np.vstack((cameraMatrix[0, 0] * x + cameraMatrix[0, 2],
                              cameraMatrix[1, 1] * y + cameraMatrix[1, 2]))
        return points2D

    def _box2cs(self, box):
        x, y, w, h = box[:4]
        return self._xywh2cs(x, y, w, h)

    def _xywh2cs(self, x, y, w, h):
        center = np.zeros((2), dtype=np.float32)
        center[0] = x + w * 0.5
        center[1] = y + h * 0.5

        if w > self.aspect_ratio * h:
            h = w * 1.0 / self.aspect_ratio
        elif w < self.aspect_ratio * h:
            w = h * self.aspect_ratio
        scale = np.array(
            [w * 1.0 / self.pixel_std, h * 1.0 / self.pixel_std],
            dtype=np.float32)
        if center[0] != -1:
            scale = scale * 1.25

        return center, scale

    def get_affine_transform(self, center, scale, rot, output_size, shift=np.array([0, 0], dtype=np.float32), inv=0):
        if not isinstance(scale, np.ndarray) and not isinstance(scale, list):
            scale = np.array([scale, scale])
        scale_tmp = scale * 200.0
        src_w = scale_tmp[0]
        dst_w = output_size[0]
        dst_h = output_size[1]
        rot_rad = np.pi * rot / 180
        src_dir = self.get_dir([0, src_w * -0.5], rot_rad)
        dst_dir = np.array([0, dst_w * -0.5], np.float32)
        src = np.zeros((3, 2), dtype=np.float32)
        dst = np.zeros((3, 2), dtype=np.float32)
        src[0, :] = center + scale_tmp * shift
        src[1, :] = center + src_dir + scale_tmp * shift
        dst[0, :] = [dst_w * 0.5, dst_h * 0.5]
        dst[1, :] = np.array([dst_w * 0.5, dst_h * 0.5]) + dst_dir
        src[2:, :] = self.get_3rd_point(src[0, :], src[1, :])
        dst[2:, :] = self.get_3rd_point(dst[0, :], dst[1, :])
        if inv:
            trans = cv2.getAffineTransform(np.float32(dst), np.float32(src))
        else:
            trans = cv2.getAffineTransform(np.float32(src), np.float32(dst))
        return trans

    def affine_transform(self, pt, t):
        new_pt = np.array([pt[0], pt[1], 1.]).T
        new_pt = np.dot(t, new_pt)
        return new_pt[:2]
    
    def get_3rd_point(self, a, b):
        direct = a - b
        return b + np.array([-direct[1], direct[0]], dtype=np.float32)
    
    def get_dir(self, src_point, rot_rad):
        sn, cs = np.sin(rot_rad), np.cos(rot_rad)
        src_result = [0, 0]
        src_result[0] = src_point[0] * cs - src_point[1] * sn
        src_result[1] = src_point[0] * sn + src_point[1] * cs
        return src_result

    def __getitem__(self, idx):
        image_path = self.imagesList[idx]
        data_numpy = cv2.imread(image_path, cv2.IMREAD_COLOR | cv2.IMREAD_IGNORE_ORIENTATION)
        data_numpy = cv2.cvtColor(data_numpy, cv2.COLOR_BGR2RGB)

        q_vbs2tango = self.frameInfoList[idx]['q_vbs2tango']
        r_Vo2To_vbs = self.frameInfoList[idx]['r_Vo2To_vbs']

        keypts2d = self.project_keypoints(q_vbs2tango, r_Vo2To_vbs,
                                          self.cameraMatrix, self.distCoeffs,
                                          self.keypts3d)  # (2, 11)

        box_full = [0, 0, self.image_width, self.image_height]

        c_full, s_full = self._box2cs(box_full)
        r_full = 0

        if self.is_train:
            sf = self.scale_factor
            rf = self.rotation_factor
            s_full = s_full * np.clip(np.random.randn()*sf + 1, 1 - sf, 1 + sf)
            r_full = np.clip(np.random.randn()*rf, -rf*2, rf*2) \
                    if random.random() <= 0.6 else 0
            
        trans_full = self.get_affine_transform(c_full, s_full, r_full, self.image_size)
        
        wrappedImage_full = cv2.warpAffine(
            data_numpy,
            trans_full,
            (int(self.image_size[0]), int(self.image_size[1])),
            flags=cv2.INTER_LINEAR)

        keypts = np.zeros_like(keypts2d)
        for i in range(keypts2d.shape[1]):
            keypts[:, i] = self.affine_transform(keypts2d[:, i], trans_full)

        xmin = float(np.min(keypts[0]))
        xmax = float(np.max(keypts[0]))
        ymin = float(np.min(keypts[1]))
        ymax = float(np.max(keypts[1]))

        xmin = max(0.0, xmin)
        ymin = max(0.0, ymin)
        xmax = min(self.image_size[0], xmax)
        ymax = min(self.image_size[1], ymax)

        w = xmax - xmin
        h = ymax - ymin
        margin_w = 0.05 * w
        margin_h = 0.05 * h
        xmin = max(0.0, xmin - margin_w)
        ymin = max(0.0, ymin - margin_h)
        xmax = min(self.image_size[0], xmax + margin_w)
        ymax = min(self.image_size[1], ymax + margin_h)

        boxTorch = torch.tensor([xmin, ymin, xmax - xmin, ymax - ymin], dtype=torch.float32)

        return wrappedImage_full, boxTorch

   
syntheticdataset = SPEEDDataset(images_dir='/kaggle/input/speedsplit/speed/images/trainval', json_dir='/kaggle/input/speedsplit/speed/val.json', tango_points_dir='/kaggle/input/mat-file/tangoPoints.mat', camera_matrix_dir='/kaggle/input/mat-file/camera.json', is_train = False)
realdataset = SPEEDDataset(images_dir='/kaggle/input/speedsplit/speed/images/real', json_dir='/kaggle/input/speedsplit/speed/real.json', tango_points_dir='/kaggle/input/mat-file/tangoPoints.mat', camera_matrix_dir='/kaggle/input/mat-file/camera.json', is_train = False)

def test_loop(test_dataloader, model, device):
    # Initialize only IoU error meter
    err_iou_meter = AverageMeter('iou')
    iou_errors_all = []

    model.eval()
    
    for batch,(X, yBbox) in enumerate(test_dataloader):
        X, yBbox = X.to(device), yBbox.to(device)
        B = X.shape[0]
        
        with torch.no_grad():
            X_np = X.cpu().numpy()
            results = model(X_np[0], verbose=False) 
         
        # IOU calculation
        if len(results) > 0 and len(results[0].boxes) > 0:
            pred_box = results[0].boxes.xyxy[0].cpu().numpy()  # [xmin, ymin, xmax, ymax]
            gt_box = yBbox[0].cpu().numpy()  # [xmin, ymin, width, height]
            gt_box_converted = [gt_box[0], gt_box[0] + gt_box[2], gt_box[1], gt_box[1] + gt_box[3]]
            
            iou = calculate_iou(gt_box_converted, pred_box)
            err_iou_meter.update(iou, 1)
            iou_errors_all.append(iou)
        else:
            iou = 0.0
            err_iou_meter.update(iou, 1)
            iou_errors_all.append(iou)
            
        print(f"\rBatch {batch+1}/{len(test_dataloader)}: IoU: {err_iou_meter.val:.2f}", end="", flush=True)

    # Simplified return structure with only IoU metrics
    performances = {
        'IoU': err_iou_meter
    }
    
    # Compute median metrics for IoU only
    medians = {
        'IoU_med': np.median(iou_errors_all) if iou_errors_all else float('nan')
    }
    
    return performances, medians
       
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = YOLO("/kaggle/input/yolospeed/runs/detect/train/weights/best.pt")

batch_size = 1

test_syn_dataloader = DataLoader(syntheticdataset, batch_size, shuffle=False, num_workers=1, pin_memory=True, drop_last=True)
test_real_dataloader = DataLoader(realdataset, batch_size, shuffle=False, num_workers=1, pin_memory=True, drop_last=True)

print("Testing on synthetic dataset...")
performances_syn, medians_syn = test_loop(test_syn_dataloader, model, device)
print("\n")
print("\nTesting on real dataset...")
performances_real, medians_real = test_loop(test_real_dataloader, model, device)
print("\n")

print("\nResults Summary:")
print("+" + "-"*22 + "+" + "-"*30 + "+" + "-"*30 + "+")
print(f"| {'Metric':<20} | {'SPEED synthetic test-set':<28} | {'SPEED real test-set':<28} |")
print("+" + "-"*22 + "+" + "-"*30 + "+" + "-"*30 + "+")
print(f"| {'Mean IoU (-)':<20} | {performances_syn['IoU'].avg: <28.4f} | {performances_real['IoU'].avg: <28.4f} |")
print(f"| {'Median IoU (-)':<20} | {medians_syn['IoU_med']: <28.4f} | {medians_real['IoU_med']: <28.4f} |")

print("+" + "-"*22 + "+" + "-"*30 + "+" + "-"*30 + "+")

100%|██████████| 5/5 [00:00<00:00, 20867.18it/s]

Testing on synthetic dataset...


Batch 2400/2400: IoU: 0.97


Testing on real dataset...
Batch 5/5: IoU: 0.93


Results Summary:
+----------------------+------------------------------+------------------------------+
| Metric               | SPEED synthetic test-set     | SPEED real test-set          |
+----------------------+------------------------------+------------------------------+
| Mean IoU (-)         | 0.9509                       | 0.9118                       |
| Median IoU (-)       | 0.9648                       | 0.9104                       |
+----------------------+------------------------------+------------------------------+
